# 05 — Descobrir o que cada componente acrescenta

Até aqui foram construídos dois mecanismos que podem mudar uma resposta: recuperação documental (etapas 01–03) e adaptador QLoRA (etapa 04). Para não atribuir um efeito ao componente errado, serão comparadas quatro condições usando a mesma base Qwen, o mesmo conjunto de perguntas e critérios comuns.

```mermaid
graph TD
 A[Qwen base] --> B[sem RAG]
 A --> C[com RAG]
 D[Qwen base + LoRA] --> E[sem RAG]
 D --> F[com RAG]
 B --> G[comparação humana]
 C --> G
 E --> G
 F --> G
```

O teste reservado não deve ser usado para escrever exemplos, ajustar parâmetros ou escolher épocas.

In [ ]:
# -----------------------------------------------------------------------------
# Objetivo desta célula: 05 — Avaliação: base, RAG e ajuste
# O código abaixo mantém as etapas explícitas para permitir inspeção de entradas,
# transformações e saídas. Conferir os valores impressos antes de avançar.
# -----------------------------------------------------------------------------
from pathlib import Path
import json

RANDOM_STATE = 42
DIRETORIO_PROJETO = Path.cwd()
if DIRETORIO_PROJETO.name != "Agente_IA_Transformers_Credito":
    DIRETORIO_PROJETO = DIRETORIO_PROJETO / "Agente_IA_Transformers_Credito"
if not DIRETORIO_PROJETO.is_dir():
    raise FileNotFoundError("Inicie o kernel na raiz do repositório ou na pasta do projeto.")

DIRETORIO_DOCUMENTOS = DIRETORIO_PROJETO / "documentos"
DIRETORIO_PROCESSADOS = DIRETORIO_PROJETO / "dados_processados"
DIRETORIO_RESULTADOS = DIRETORIO_PROJETO / "resultados"
DIRETORIO_PROCESSADOS.mkdir(exist_ok=True)
DIRETORIO_RESULTADOS.mkdir(exist_ok=True)
print("Projeto:", DIRETORIO_PROJETO)

## 1. Escolher uma condição por execução

A GPU tem memória limitada, então cada execução carrega a base ou a base com adaptador, nunca ambas simultaneamente. Em cada modo são geradas respostas sem RAG e com RAG. Repetir em kernel limpo para o modo base e para o ajustado produz as quatro condições comparáveis.

O código valida se o modo declarado corresponde aos pesos carregados. Essa checagem evita rotular uma resposta base como ajustada ou vice-versa.

---

### Por que quatro condições?

Há dois fatores: ajuste (base ou base + LoRA) e recuperação (sem ou com RAG). O arranjo 2 × 2 permite observar o efeito de cada fator e se eles se complementam. Para atribuir diferenças ao ajuste, comparam-se base e ajustado mantendo RAG igual; para medir RAG, comparam-se sem e com RAG mantendo os pesos iguais.

Exemplo puramente ilustrativo, em escala de qualidade 1–5: base sem RAG = 2, base com RAG = 3, ajustado sem RAG = 3, ajustado com RAG = 4. A leitura sugeriria +1 para cada fator nesse exemplo, mas os valores não são resultados deste projeto. Respostas reais precisam ser avaliadas por pergunta com uma rubrica definida.

In [ ]:
# -----------------------------------------------------------------------------
# Objetivo desta célula: Pergunta de aprendizagem: qual transformação ou decisão este bloco permite compreender?
# O código abaixo mantém as etapas explícitas para permitir inspeção de entradas,
# transformações e saídas. Conferir os valores impressos antes de avançar.
# -----------------------------------------------------------------------------
import time
import chromadb
from sentence_transformers import SentenceTransformer
from rag import ENCODER, COLECAO, carregar_llm, recuperar, montar_mensagens, gerar

MODO_AVALIACAO = "base"  # Trocar para "ajustado" em um novo kernel.
adaptador = DIRETORIO_PROJETO / "adaptador_pf" if MODO_AVALIACAO == "ajustado" else None
if adaptador is not None and not adaptador.is_dir():
    raise FileNotFoundError("Execute o notebook 04 antes de avaliar o adaptador.")
modelo, tokenizador = carregar_llm(adaptador)
encoder = SentenceTransformer(ENCODER, device="cpu")
colecao = chromadb.PersistentClient(path=str(DIRETORIO_PROJETO / "banco_chroma")).get_collection(COLECAO)
exemplos = [json.loads(l) for l in (DIRETORIO_PROJETO / "dataset_sft.jsonl").read_text(encoding="utf-8").splitlines() if l]
teste = [e for e in exemplos if e["split"] == "teste"]

## 2. Executar perguntas reservadas e guardar respostas

As perguntas de teste devem cobrir conceitos, matemática, código e reconhecimento de evidência insuficiente. Para as condições com RAG, usa-se o mesmo índice e o mesmo processo de recuperação; assim, mudanças entre base e ajustado podem ser atribuídas com mais clareza ao gerador. Para separar recuperação de geração, também é útil comparar respostas com trechos fixos.

As saídas são salvas para revisão humana. Use uma rubrica comum para correção, fidelidade às fontes, validade de citações, clareza didática, utilidade das propostas e reconhecimento de limites.

---

### Rubrica e exemplo de avaliação

Cada resposta pode receber notas separadas para correção, cobertura, fidelidade às fontes, citação e clareza. Exemplo: uma resposta conceitualmente correta que cita a página errada pode receber boa nota de correção e baixa nota de citação; uma nota única esconderia esse tipo de falha.

As mesmas perguntas e critérios devem ser usados nas quatro condições. Ao revisar, registrar também a causa provável: extração, chunking, recuperação, geração, citação ou contexto insuficiente. Isso indica qual etapa deve ser melhorada.

In [ ]:
# -----------------------------------------------------------------------------
# Objetivo desta célula: Parte VI — Avaliar e entregar
# O código abaixo mantém as etapas explícitas para permitir inspeção de entradas,
# transformações e saídas. Conferir os valores impressos antes de avançar.
# -----------------------------------------------------------------------------
EXECUTAR_AVALIACAO = False
if EXECUTAR_AVALIACAO:
    assert (MODO_AVALIACAO == "ajustado") == hasattr(modelo, "peft_config")
    avaliacoes = []
    for exemplo in teste:
        for usar_rag in [False, True]:
            evidencias = recuperar(colecao, encoder, exemplo["pergunta"]) if usar_rag else []
            mensagens = montar_mensagens(exemplo["pergunta"], evidencias) if usar_rag else [
                {"role": "system", "content": INSTRUCAO_SISTEMA},
                {"role": "user", "content": exemplo["pergunta"]},
            ]
            inicio = time.perf_counter()
            resposta_teste = gerar(modelo, tokenizador, mensagens)
            avaliacoes.append({"modo": MODO_AVALIACAO, "rag": usar_rag, "pergunta": exemplo["pergunta"], "referencia": exemplo["resposta"], "resposta": resposta_teste, "fontes": evidencias, "tempo_s": time.perf_counter()-inicio})
    (DIRETORIO_RESULTADOS / f"avaliacao_{MODO_AVALIACAO}.json").write_text(json.dumps(avaliacoes, ensure_ascii=False, indent=2), encoding="utf-8")

## 3. Fechar a trilha e decidir o próximo ciclo

A sequência construída é: entrada e geração (00) → documentos e embeddings (01) → índice e ranking (02) → resposta com evidências (03) → adaptação supervisionada (04) → comparação controlada (05). Cada artefato alimenta o próximo, e os manifestos/arquivos permitem reabrir etapas em kernels limpos.

O README deve registrar o que foi realmente executado, quais perguntas foram avaliadas, limites encontrados e o próximo ajuste justificável pelos erros. A expansão do corpus ou do dataset deve responder a uma lacuna observada, mantendo procedência e revisão humana.

---

### Limite da conclusão

Se o conjunto de teste tiver poucas perguntas, uma diferença pequena pode depender de um ou dois casos. Exemplo: em dez perguntas, uma resposta adicional avaliada como correta altera a taxa simples de acerto em 10 pontos percentuais. Por isso, mostrar contagens, exemplos e resultados por tema, além de médias; não extrapolar para desempenho operacional de crédito.